# 📄 Leer formularios GRH-006 con PP-StructureV3

Extrae los datos de los formularios **GRH-006** (SOLICITUD DE NECESIDADES DE CAPACITACIÓN Y DESARROLLO)
del CIGB usando **PaddleOCR v3.7.0 / PP-StructureV3**.

**Columnas extraídas**: Solapín | Nombre y Apellidos | Actividad

---
### ⚙️ Cómo usar:
1. **Ejecutar celda por celda** (Play ▶️ en cada una)
2. La primera celda installa las dependencias
3. Después subís los PDFs o los conectás desde Google Drive
4. Al final descargás los resultados en Excel

> Hecho por doclens — Reutilizable todos los años 🙌

---
## 1. Instalar dependencias

In [ ]:
# @title Instalar PaddleOCR + dependencias

# Instalar PaddlePaddle (GPU) y PaddleOCR
!pip install -q paddlepaddle-gpu paddleocr

# Para exportar a Excel
!pip install -q openpyxl pandas

print("✅ Dependencias instaladas")

In [ ]:
# Verificar instalación
import paddle
print(f"PaddlePaddle: {paddle.__version__}")
print(f"GPU disponible: {paddle.is_compiled_with_cuda()}")

from paddleocr import PPStructure
from paddleocr import __version__ as pocr_version
print(f"PaddleOCR: {pocr_version}")

import pandas as pd
print(f"Pandas: {pd.__version__}")

---
## 2. Cargar los PDFs

Elegí una opción:
- **A** — Subir archivos manualmente a Colab
- **B** — Conectar Google Drive (recomendado para uso anual)

In [ ]:
# @title Conectar Google Drive (o elegir carpeta local)

import os

# Opción A: Subir archivos manualmente
from google.colab import files

USAR_DRIVE = False  # Cambiar a True para usar Google Drive

if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Configurar ruta a la carpeta con los PDFs
    PDF_DIR = '/content/drive/MyDrive/DNC 2026'  # CAMBIAR esta ruta
else:
    print("💡 Subí los PDFs con el botón 'Choose Files' que aparece abajo:")
    uploaded = files.upload()
    PDF_DIR = '/content'
    print(f"\n✅ {len(uploaded)} archivos subidos")

# Listar PDFs encontrados
pdfs = [f for f in os.listdir(PDF_DIR) if f.lower().endswith('.pdf')]
print(f"\n📂 PDFs encontrados: {len(pdfs)}")
for p in sorted(pdfs):
    print(f"   - {p}")

---
## 3. Cargar PP-StructureV3

Este modelo analiza la estructura del documento y extrae tablas y texto.

In [ ]:
# @title Inicializar PP-StructureV3
from paddleocr import PPStructure
import json

print("🔄 Cargando PP-StructureV3...")
engine = PPStructure(
    show_log=False,
    lang='es',  # Español
    use_gpu=True,
    det_db_thresh=0.3,
    det_db_box_thresh=0.5
)
print("✅ Modelo cargado correctamente")

---
## 4. Extraer datos de todos los PDFs

Esto puede tardar varios minutos dependiendo de la cantidad de PDFs.

In [ ]:
# @title Procesar PDFs y extraer tablas
import time

resultados_totales = []
errores = []

pdfs_a_procesar = sorted([f for f in os.listdir(PDF_DIR) if f.lower().endswith('.pdf')])
total = len(pdfs_a_procesar)

print(f"📄 Procesando {total} PDFs...\n")

for idx, pdf_name in enumerate(pdfs_a_procesar, 1):
    pdf_path = os.path.join(PDF_DIR, pdf_name)
    print(f"  [{idx}/{total}] {pdf_name}...", end=' ', flush=True)

    try:
        t0 = time.time()
        result = engine.predict(pdf_path)
        elapsed = time.time() - t0

        datos_pdf = {
            'archivo': pdf_name,
            'paginas': len(result),
            'tiempo_seg': round(elapsed, 1),
            'tablas': []
        }

        for page_idx, page_result in enumerate(result):
            for element in page_result:
                if element.get('type') == 'table':
                    html_table = element.get('res', {})

                    # Extraer celdas del HTML de la tabla
                    import re
                    rows_data = []
                    # Buscar filas en el HTML
                    row_matches = re.findall(r'<tr>(.*?)</tr>', html_table, re.DOTALL)
                    for row_html in row_matches:
                        cells = re.findall(r'<td[^>]*>(.*?)</td>', row_html, re.DOTALL)
                        # Limpiar etiquetas HTML y espacios
                        cells_clean = []
                        for c in cells:
                            c_clean = re.sub(r'<[^>]+>', '', c).strip()
                            cells_clean.append(c_clean)
                        if cells_clean:
                            rows_data.append(cells_clean)

                    datos_pdf['tablas'].append({
                        'pagina': page_idx + 1,
                        'filas': len(rows_data),
                        'datos': rows_data
                    })

        resultados_totales.append(datos_pdf)
        print(f"✅ {elapsed:.1f}s | {datos_pdf['paginas']} pág(s)")

        # Mostrar preview de la tabla si se encontró
        if datos_pdf['tablas']:
            for t in datos_pdf['tablas']:
                if t['filas'] > 1:
                    print(f"     ↳ Tabla en pág {t['pagina']}: {t['filas']} filas")
                    for row in t['datos'][:3]:  # primeras 3 filas
                        print(f"       {row}")
                    if t['filas'] > 3:
                        print(f"       ... y {t['filas'] - 3} filas más")

    except Exception as e:
        print(f"❌ ERROR: {e}")
        errores.append({'archivo': pdf_name, 'error': str(e)})

print(f"\n{'='*50}")
print(f"✅ Procesados: {len(resultados_totales)} / {total}")
print(f"❌ Errores: {len(errores)}")
print(f"📊 Total tablas encontradas: {sum(len(r['tablas']) for r in resultados_totales)}")

---
## 5. Resultados

Armamos un DataFrame con todos los datos extraídos y exportamos a Excel.

In [ ]:
# @title Armar tabla consolidada

filas_consolidadas = []

for pdf_result in resultados_totales:
    archivo = pdf_result['archivo']
    for tabla in pdf_result['tablas']:
        pagina = tabla['pagina']
        datos = tabla['datos']

        for fila_idx, fila in enumerate(datos):
            # Intentar mapear columnas según la estructura GRH-006
            # Generalmente: [Solapín, Nombre, Actividad] o variantes
            if len(fila) >= 3:
                filas_consolidadas.append({
                    'Archivo': archivo,
                    'Pagina': pagina,
                    'Fila': fila_idx + 1,
                    'Col1': fila[0] if len(fila) > 0 else '',
                    'Col2': fila[1] if len(fila) > 1 else '',
                    'Col3': fila[2] if len(fila) > 2 else '',
                    'Col4': fila[3] if len(fila) > 3 else '',
                    'Col5': fila[4] if len(fila) > 4 else '',
                })
            else:
                # Columnas variables
                row_data = {'Archivo': archivo, 'Pagina': pagina, 'Fila': fila_idx + 1}
                for ci, c in enumerate(fila):
                    row_data[f'Col{ci+1}'] = c
                filas_consolidadas.append(row_data)

df = pd.DataFrame(filas_consolidadas)

print(f"📋 Total filas extraídas: {len(df)}")
print(f"📋 Total PDFs con datos: {df['Archivo'].nunique()}")
print(f"\n🔍 Preview (primeras 10 filas):")
display(df.head(10))

In [ ]:
# @title Exportar a Excel
from datetime import datetime

fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
excel_name = f'GRH006_extraccion_{fecha}.xlsx'

# Guardar Excel
df.to_excel(excel_name, index=False, engine='openpyxl')

print(f"✅ Excel guardado: {excel_name}")

# Descargar
from google.colab import files
files.download(excel_name)

print("📥 Descarga iniciada...")

In [ ]:
# @title Reporte de errores (si hubo)

if errores:
    df_errores = pd.DataFrame(errores)
    print(f"❌ {len(errores)} PDF(s) con errores:")
    display(df_errores)
else:
    print("✅ Ningún error durante el procesamiento")

---
## ✅ Listo

Los datos de los formularios GRH-006 fueron extraídos.

### Próximos pasos posibles:
1. **Revisar la calidad** de los datos extraídos
2. **Afinar columnas** si la estructura de la tabla no se mapea bien
3. **Agregar corrección** con un modelo de lenguaje si hay errores OCR

### Para usar el próximo año:
- Solo cambiá los PDFs en la misma carpeta de Drive y ejecutá de nuevo
- No hace falta recargar el modelo (solo saltar a la sección 4)

---
📁 **doclens** — Pipeline de extracción de formularios escaneados